# Convolution and Translation Equivariance

AI & ML Foundations: Convolutional networks · *Mathematics for AI and Machine Learning* by Fuheng Wu

Shared weights make convolution commute with shifts; per-position weights break it (Chapter 6).

Companion code for the [interactive demo](https://math4ai.org/companion/ml/conv-equivariance/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch
import torch.nn.functional as F

# The demo's signal: a box at positions 8-13 and a spike at 26, length 40; kernel [-1, 0, 1]
N = 40
x = torch.zeros(N)
x[8:14] = 1.0
x[26] = 1.5
w = torch.tensor([-1.0, 0.0, 1.0])


def conv(x, w):
    """Circular 1D convolution (cross-correlation, as in deep learning)."""
    xp = F.pad(x[None, None], (1, 1), mode="circular")
    return F.conv1d(xp, w[None, None])[0, 0]


def locally_connected(x, w):
    """Same window, but different weights at every position (no weight sharing)."""
    scale = 1 + 0.8 * torch.sin(torch.arange(N) / 3)
    xp = torch.cat([x[-1:], x, x[:1]])
    windows = xp.unfold(0, 3, 1)                 # (N, 3)
    return (windows * w * scale[:, None]).sum(1)


# ── Equivariance: conv(shift x) == shift(conv x) for every shift ────────────
for s in (1, 3, 7, -5):
    a = conv(torch.roll(x, s), w)
    b = torch.roll(conv(x, w), s)
    c = locally_connected(torch.roll(x, s), w)
    d = torch.roll(locally_connected(x, w), s)
    print(f"shift {s:+d}: convolution difference {(a - b).abs().max().item():.1e} | "
          f"locally connected difference {(c - d).abs().max().item():.3f}")

# ── Convolution is a circulant matrix, and circulant matrices commute with shifts
C = torch.stack([conv(e, w) for e in torch.eye(N)], 1)            # column j = response to e_j
S = torch.roll(torch.eye(N), 1, dims=0)                           # the shift-by-one matrix
print("\nconvolution matrix C commutes with the shift S (C S == S C):", torch.allclose(C @ S, S @ C))
print("every row of C is the previous row shifted (circulant):", torch.allclose(torch.roll(C, (1, 1), dims=(0, 1)), C))

# ── Max pooling (window 4) turns shifts by 4 into shifts by 1 of the output ─
pool = lambda y: F.max_pool1d(y[None, None], 4)[0, 0]
print("pool(conv(shift_4 x)) == shift_1 pool(conv x):", torch.allclose(pool(conv(torch.roll(x, 4), w)), torch.roll(pool(conv(x, w)), 1)))
print("pool(conv(shift_1 x)) == pool(conv x):        ", torch.allclose(pool(conv(torch.roll(x, 1), w)), pool(conv(x, w))),
      "(pooling is only approximately invariant)")

shift +1: convolution difference 0.0e+00 | locally connected difference 0.385
shift +3: convolution difference 0.0e+00 | locally connected difference 1.147
shift +7: convolution difference 0.0e+00 | locally connected difference 2.200
shift -5: convolution difference 0.0e+00 | locally connected difference 1.109

convolution matrix C commutes with the shift S (C S == S C): True
every row of C is the previous row shifted (circulant): True
pool(conv(shift_4 x)) == shift_1 pool(conv x): True
pool(conv(shift_1 x)) == pool(conv x):         False (pooling is only approximately invariant)
